# Spell Feature Engineering

This notebook extracts features from D&D 5e spell data.

**Input:** `data/spells.csv`
**Output:** Updated `data/spells.csv` with additional feature columns

## Imports and Setup

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Detect execution context
cwd = Path.cwd()
if cwd.name == 'notebooks':
    DATA_DIR = '../data'
    sys.path.insert(0, '.')
    from helper_files import parse_spell_targets, calculate_average_damage
else:
    DATA_DIR = './data'
    sys.path.insert(0, '.')
    from notebooks.helper_files import parse_spell_targets, calculate_average_damage

print(f"Data directory: {DATA_DIR}")
print("Imports successful")

Data directory: ./data
Imports successful


## Load Spell Data

In [2]:
load_path = DATA_DIR + '/spells.csv'
df = pd.read_csv(load_path)

In [3]:
# Parse target information for each spell
target_data = df['description'].apply(parse_spell_targets)

# Extract individual columns from the dict results
df['target_count'] = target_data.apply(lambda x: x['target_count'])
df['is_aoe'] = target_data.apply(lambda x: x['is_aoe'])
df['aoe_type'] = target_data.apply(lambda x: x['aoe_type'])
df['aoe_size'] = target_data.apply(lambda x: x['aoe_size'])
df['estimated_targets'] = target_data.apply(lambda x: x['estimated_targets'])

# For non-AoE spells with target_count, use that as estimated_targets
df.loc[df['target_count'].notna() & df['estimated_targets'].isna(), 'estimated_targets'] = df['target_count']

print("Target features extracted")

Target features extracted


## Extract Target Information

Using `parse_spell_targets()` to extract:
- `target_count`: Number of targets (1, 2, 3, etc.) or None for AoE
- `is_aoe`: Whether the spell is Area of Effect
- `aoe_type`: Type of AoE (cone, line, radius, cube, sphere)
- `aoe_size`: Size of the AoE (e.g., "20-foot")

In [4]:
# Parse target information for each spell
target_data = df['description'].apply(parse_spell_targets)

# Extract individual columns from the dict results
df['target_count'] = target_data.apply(lambda x: x['target_count'])
df['is_aoe'] = target_data.apply(lambda x: x['is_aoe'])
df['aoe_type'] = target_data.apply(lambda x: x['aoe_type'])
df['aoe_size'] = target_data.apply(lambda x: x['aoe_size'])

print("Target features extracted")

Target features extracted


## Target Analysis

In [5]:
# Check specific spells to verify parsing
examples = [
    'Magic Missile',      # Should be 3 targets (darts)
    'Fireball',           # Should be AoE radius ~7.5 targets
    'Cure Wounds',        # Should be 1 target
    'Scorching Ray',      # Should be 3 targets (rays)
    'Lightning Bolt',     # Should be AoE line ~3 targets
    'Cone of Cold',       # Should be AoE cone ~10.8 targets
    'Hold Person',        # Should be 1 target
    'Chain Lightning',    # Should be multi-target or AoE
    'Eldritch Blast',     # Should be 1 target (at base level)
    'Burning Hands',      # Should be AoE cone (small)
]

print("=== Example Spells ===")
for spell_name in examples:
    spell = df[df['spell_name'] == spell_name]
    if len(spell) > 0:
        row = spell.iloc[0]
        if row['is_aoe']:
            print(f"{spell_name}: AoE {row['aoe_type']} ({row['aoe_size']}) → ~{row['estimated_targets']} targets")
        elif pd.notna(row['target_count']):
            print(f"{spell_name}: {int(row['target_count'])} target(s)")
        else:
            print(f"{spell_name}: unknown targeting")

=== Example Spells ===
Magic Missile: 3 target(s)
Fireball: AoE radius (20-foot) → ~7.5 targets
Cure Wounds: 1 target(s)
Scorching Ray: 3 target(s)
Lightning Bolt: AoE line (100-foot) → ~3.0 targets
Cone of Cold: AoE cone (60-foot) → ~10.8 targets
Hold Person: 1 target(s)
Chain Lightning: 3 target(s)
Eldritch Blast: 1 target(s)
Burning Hands: AoE cone (15-foot) → ~1.0 targets


In [6]:
# AoE type breakdown
print("\n=== AoE Types ===")
aoe_spells = df[df['is_aoe']]
print(aoe_spells['aoe_type'].value_counts())


=== AoE Types ===
aoe_type
radius    80
cube      35
cone      17
area      12
line       9
square     8
Name: count, dtype: int64


In [7]:
# Target count distribution (non-AoE)
print("\n=== Target Count Distribution (non-AoE) ===")
non_aoe = df[~df['is_aoe'] & df['target_count'].notna()]
print(non_aoe['target_count'].value_counts().sort_index())


=== Target Count Distribution (non-AoE) ===
target_count
1.0     316
2.0       1
3.0       8
4.0       2
5.0       4
6.0       3
8.0       4
10.0      6
Name: count, dtype: int64


## Verify Examples

In [8]:
# Check specific spells to verify parsing
examples = [
    'Magic Missile',      # Should be 3 targets (darts)
    'Fireball',           # Should be AoE radius
    'Cure Wounds',        # Should be 1 target
    'Scorching Ray',      # Should be 3 targets (rays)
    'Lightning Bolt',     # Should be AoE line
    'Cone of Cold',       # Should be AoE cone
    'Hold Person',        # Should be 1 target
    'Chain Lightning',    # Should be multi-target or AoE
    'Eldritch Blast',     # Should be 1 target (at base level)
]

print("=== Example Spells ===")
for spell_name in examples:
    spell = df[df['spell_name'] == spell_name]
    if len(spell) > 0:
        row = spell.iloc[0]
        if row['is_aoe']:
            print(f"{spell_name}: AoE ({row['aoe_type']}, {row['aoe_size']})")
        elif pd.notna(row['target_count']):
            print(f"{spell_name}: {int(row['target_count'])} target(s)")
        else:
            print(f"{spell_name}: unknown targeting")

=== Example Spells ===
Magic Missile: 3 target(s)
Fireball: AoE (radius, 20-foot)
Cure Wounds: 1 target(s)
Scorching Ray: 3 target(s)
Lightning Bolt: AoE (line, 100-foot)
Cone of Cold: AoE (cone, 60-foot)
Hold Person: 1 target(s)
Chain Lightning: 3 target(s)
Eldritch Blast: 1 target(s)


# Top single-target damage spells


In [9]:
damage_spells = df[df['avg_damage']>0].reset_index(drop=True)

In [10]:
print("\\n=== Top 10 Single-Target Damage Spells ===")
top_single = damage_spells[damage_spells['target_count'] == 1].nlargest(10, 'avg_damage')
print(top_single[['spell_name', 'level', 'damage_dice', 'avg_damage']].to_string(index=False))

\n=== Top 10 Single-Target Damage Spells ===
                    spell_name  level damage_dice  avg_damage
                   Time Ravage      9       10d12        65.0
               Finger of Death      7    7d8 + 30        61.5
                          Harm      6        14d6        49.0
            Psychic Crush (UA)      6        12d6        42.0
                 Reality Break      8        6d12        39.0
                        Blight      4         8d8        36.0
Raulothim's Psychic Lance (UA)      4        10d6        35.0
                 Blade Barrier      6        6d10        33.0
         Negative Energy Flood      5        5d12        32.5
               Banishing Smite      5        5d10        27.5


In [11]:
# Calculate total estimated damage (damage × estimated targets)
damage_spells['total_estimated_damage'] = damage_spells['avg_damage'] * damage_spells['estimated_targets']

print("\n=== Top 10 Spells by Total Estimated Damage ===")
print("(avg_damage × estimated_targets)")
top_total = damage_spells.dropna(subset=['total_estimated_damage']).nlargest(10, 'total_estimated_damage')
print(top_total[['spell_name', 'level', 'damage_dice', 'avg_damage', 'estimated_targets', 'total_estimated_damage']].to_string(index=False))


=== Top 10 Spells by Total Estimated Damage ===
(avg_damage × estimated_targets)
               spell_name  level damage_dice  avg_damage  estimated_targets  total_estimated_damage
                   Symbol      7       10d10        55.0               67.9                 3734.50
               Earthquake      8         5d6        17.5              188.5                 3298.75
                 Sunburst      8        12d6        42.0               67.9                 2851.80
       Maddening Darkness      8         8d8        36.0               67.9                 2444.40
Otiluke's Freezing Sphere      6        10d6        35.0               67.9                 2376.50
             Meteor Swarm      9        20d6        70.0               30.2                 2114.00
          Circle of Death      6         8d6        28.0               67.9                 1901.20
           Call Lightning      3        3d10        16.5               67.9                 1120.35
           Conjure

In [12]:
# Combine damage and target info for damage spells
damage_spells = df[df['avg_damage'] > 0].copy()

print(f"=== Damage Spells: {len(damage_spells)} ===")
print(f"\nAoE damage spells: {damage_spells['is_aoe'].sum()}")
print(f"Single-target damage: {(damage_spells['target_count'] == 1).sum()}")
print(f"Multi-target damage: {((damage_spells['target_count'] > 1) & ~damage_spells['is_aoe']).sum()}")

=== Damage Spells: 210 ===

AoE damage spells: 97
Single-target damage: 94
Multi-target damage: 7


In [13]:
# Final summary
print("\n=== Final Summary ===")
print(f"Total spells: {len(df)}")
print(f"With damage: {(df['avg_damage'] > 0).sum()}")
print(f"AoE: {df['is_aoe'].sum()}")
print(f"Single-target: {(df['target_count'] == 1).sum()}")
print(f"Multi-target (non-AoE): {((df['target_count'] > 1) & ~df['is_aoe']).sum()}")
print(f"\nNew columns added: target_count, is_aoe, aoe_type, aoe_size, estimated_targets")


=== Final Summary ===
Total spells: 574
With damage: 210
AoE: 161
Single-target: 316
Multi-target (non-AoE): 28

New columns added: target_count, is_aoe, aoe_type, aoe_size, estimated_targets


In [14]:
# Top AoE damage spells
print("\n=== Top 10 AoE Damage Spells ===")
top_aoe = damage_spells[damage_spells['is_aoe']].nlargest(10, 'avg_damage')
print(top_aoe[['spell_name', 'level', 'aoe_type', 'aoe_size', 'damage_dice', 'avg_damage']].to_string(index=False))


=== Top 10 AoE Damage Spells ===
                 spell_name  level aoe_type aoe_size damage_dice  avg_damage
               Disintegrate      6     cube  10-foot   10d6 + 40        75.0
               Meteor Swarm      9   radius  40-foot        20d6        70.0
                     Symbol      7   radius  60-foot       10d10        55.0
Abi-Dalzim's Horrid Wilting      8     cube  30-foot        12d8        54.0
           Incendiary Cloud      8   radius  20-foot        10d8        45.0
     Delayed Blast Fireball      7   radius  20-foot        12d6        42.0
                   Sunburst      8   radius  60-foot        12d6        42.0
                 Fire Storm      7     cube  10-foot        7d10        38.5
               Cone of Cold      5     cone  60-foot         8d8        36.0
             Conjure Volley      5   radius  40-foot         8d8        36.0


In [15]:
# Top single-target damage spells
print("\n=== Top 10 Single-Target Damage Spells ===")
top_single = damage_spells[damage_spells['target_count'] == 1].nlargest(10, 'avg_damage')
print(top_single[['spell_name', 'level', 'damage_dice', 'avg_damage']].to_string(index=False))


=== Top 10 Single-Target Damage Spells ===
                    spell_name  level damage_dice  avg_damage
                   Time Ravage      9       10d12        65.0
               Finger of Death      7    7d8 + 30        61.5
                          Harm      6        14d6        49.0
            Psychic Crush (UA)      6        12d6        42.0
                 Reality Break      8        6d12        39.0
                        Blight      4         8d8        36.0
Raulothim's Psychic Lance (UA)      4        10d6        35.0
                 Blade Barrier      6        6d10        33.0
         Negative Energy Flood      5        5d12        32.5
               Banishing Smite      5        5d10        27.5


# Save Updated Data

In [16]:
# Save the updated dataframe
output_path = f"{DATA_DIR}/spells.csv"
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} spells to {output_path}")
print(f"\nColumns: {list(df.columns)}")

Saved 574 spells to ./data/spells.csv

Columns: ['spell_name', 'source', 'level', 'school', 'casting_time', 'range', 'components', 'duration', 'description', 'level_scaling', 'spell_lists', 'damage_dice', 'damage_type', 'avg_damage', 'target_count', 'is_aoe', 'aoe_type', 'aoe_size', 'estimated_targets']


In [17]:
# Final summary
print("\n=== Final Summary ===")
print(f"Total spells: {len(df)}")
print(f"With damage: {(df['avg_damage'] > 0).sum()}")
print(f"AoE: {df['is_aoe'].sum()}")
print(f"Single-target: {(df['target_count'] == 1).sum()}")
print(f"Multi-target: {(df['target_count'] > 1).sum()}")


=== Final Summary ===
Total spells: 574
With damage: 210
AoE: 161
Single-target: 316
Multi-target: 28


# My Analysis